# 365 Probabilidades - Dia #104
## Qual a probabilidade de a Mega-Sena lembrar quais números já saíram?

**Tipo:** Cabalístico  
**Data de publicação:** 2026-09-25  
**Ferramenta:** Python  
**Decisão analisada:** Escolher os números da aposta pela lista dos "atrasados" ou dos "mais sorteados"  
**Hashtag:** #365Probabilidades #Dia104

---

### 📖 A História

Todo mundo já fez isso pelo menos uma vez.

Abriu a lista dos números mais atrasados da Mega-Sena, viu uma dezena que não sai há meses e sentiu, com toda a convicção do mundo, que a vez dela estava chegando.

Ou fez o contrário. Foi na lista dos mais sorteados, porque se um número sai tanto assim, alguma coisa ele deve ter.

As duas listas estão em qualquer site de resultado. E as duas partem da mesma crença: a de que o globo, de algum jeito, se lembra do que já tirou.

Hoje eu fui conferir essa memória com o histórico da Mega-Sena, do primeiro sorteio de 1996 até maio de 2026.

---

### 📚 O Conceito: a bola sem memória

Em cada concurso saem 6 dezenas de 60. Para qualquer dezena, a chance de estar entre elas é **6 em 60, ou seja, 10%**.

Se os concursos são independentes, essa chance não depende do que aconteceu antes. É isso que se chama **ausência de memória**: ficar 40 concursos sem sair não aumenta nem diminui a chance do 41º.

Daí sai a assinatura matemática do dia. A chance de uma dezena ficar pelo menos *k* concursos sem sair é

$$P(\text{atraso} \geq k) = 0{,}9^{k}$$

e a chance de ela sair no próximo concurso, qualquer que seja o atraso, é sempre a mesma:

$$P(\text{sair} \mid \text{atraso} = k) = \frac{6}{60} = 10\%$$

Acreditar que o atrasado "está devendo" é a **falácia do jogador**. Tversky e Kahneman a descreveram em 1971 como parte de uma crença mais ampla: a de que sequências curtas precisam se parecer com a média de longo prazo, como se o acaso corrigisse os próprios desvios.

Existe um segundo engano, mais sutil. Com 60 dezenas e três mil concursos, **sempre** vai haver uma campeã, uma lanterna e uma dezena muito atrasada. Uma lista de mais sorteados não prova nada sozinha. A pergunta certa é se ela é mais desigual do que um sorteio justo produziria.

---

### 🧮 O Modelo

1. **A teoria:** a conta exata do atraso esperado para um sorteio sem memória.
2. **A memória do atraso:** para cada dezena, em cada concurso, quanto tempo ela estava sem sair e se saiu no concurso seguinte. Taxa de saída por faixa de atraso, com IC 95% pela Beta de Jeffreys. O mesmo teste para as dezenas "quentes", pela quantidade de vezes em que saíram nos últimos 20 concursos.
3. **O globo é justo?** Teste de uniformidade das 60 dezenas na série inteira e separadamente nas duas eras de equipamento (duas gaiolas até o concurso 1.139, globo único a partir do 1.140). A divisão por era foi definida antes de rodar.
4. **O acaso parece arrumado:** 2.000 históricos simulados de sorteio justo, com o mesmo número de concursos, para saber que campeã, que lanterna e que atraso o acaso puro produz.
5. **Três jogadores:** quem jogou sempre nas 6 mais atrasadas, quem jogou nas 6 mais sorteadas até então e quem usou a Surpresinha, concurso a concurso, a história inteira.

Todos os p-valores vêm de **Monte Carlo do mecanismo real** (6 de 60 sem reposição por concurso), não da tabela do qui-quadrado, porque as 6 dezenas de um mesmo concurso não são independentes entre si.

**Fontes:**
- Resultados da Mega-Sena, concursos 1 (11/03/1996) a 3.005 (07/05/2026). Cópia pública dos resultados da Caixa, em repositório no GitHub, no mesmo formato do arquivo oficial. O arquivo acompanha o notebook.
- Caixa Econômica Federal, Portal Loterias. Regras vigentes: 6 dezenas de 60, chance de 1 em 50.063.860 na aposta simples de R$ 6,00, prêmio bruto de 43,79% da arrecadação, sorteios às terças, quintas e domingos.
- Wikipédia (pt), verbete Mega-Sena. Troca das duas gaiolas pelo globo único a partir do concurso 1.140. Não confirmado em documento da Caixa.
- Tversky, A. & Kahneman, D. (1971). Belief in the law of small numbers. *Psychological Bulletin*, 76(2), 105-110. Referência conceitual.

**Fator ×0,80:** não aplicado. Os dados são o registro administrativo completo dos sorteios, não autorrelato.


In [ ]:
import unicodedata, re
from math import comb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEED = 42
rng = np.random.default_rng(SEED)

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

DOURADO, VERMELHO, VERDE, CINZA = '#c9a14a', '#c0392b', '#2a8a82', '#6b6a64'

print("✅ Bibliotecas carregadas · semente", SEED)

In [ ]:
# --- DADOS ---

# =========================================================================
# 1) Resultados da Mega-Sena, concursos 1 a 3.005 (11/03/1996 a 07/05/2026)
#    Arquivo: megasena_caixa_ate_3005.csv, na mesma pasta do notebook.
#    Origem: cópia pública dos resultados da Caixa (repositório no GitHub),
#    no mesmo formato do arquivo oficial. Não é o download oficial do
#    Portal Loterias Caixa: ver Limitações. O notebook não baixa nada.
#    Aceita .csv ou .xlsx, então trocar pelo oficial é só mudar esta linha.
# =========================================================================
ARQUIVO = 'megasena_caixa_ate_3005.csv'
CORTE = 3005      # último concurso incluído (None = todos os do arquivo)

# 2) Regras vigentes (Portal Loterias Caixa, setembro de 2026)
DEZENAS, SORTEADAS = 60, 6
P_DEZENA = SORTEADAS / DEZENAS            # 0,10 por concurso, para qualquer dezena
COMBINACOES = comb(DEZENAS, SORTEADAS)    # 50.063.860
APOSTA_SIMPLES = 6.00                     # R$ 6,00 por 6 dezenas
PREMIO_BRUTO = 0.4379                     # prêmio bruto = 43,79% da arrecadação

# 3) Mudança do equipamento: duas gaiolas (0-5 e 0-9, 00 = 60) até o concurso
#    1.139; globo único com bolas de 01 a 60 a partir do 1.140.
#    Fonte: Wikipédia (pt). Não confirmado em documento da Caixa.
CONCURSO_GLOBO_UNICO = 1140

# 4) Parâmetros do modelo (definidos antes de rodar)
BURN = 20          # concursos iniciais usados só para montar o histórico
JANELA = 20        # "quente" = quantas vezes saiu nos últimos 20 concursos
FAIXAS_ATRASO = [0, 5, 10, 20, 30, 50, np.inf]
ROT_ATRASO = ['0-4', '5-9', '10-19', '20-29', '30-49', '50+']
FAIXAS_QUENTE = [0, 1, 2, 3, 4, np.inf]
ROT_QUENTE = ['0', '1', '2', '3', '4+']
R_SIM = 2000       # históricos simulados no Monte Carlo


def mil(n):
    return f"{int(n):,}".replace(',', '.')


def _norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]', '', s.lower())


def carregar(caminho):
    if caminho.lower().endswith(('.xlsx', '.xls')):
        df = pd.read_excel(caminho)
    else:
        df = pd.read_csv(caminho, sep=None, engine='python', encoding='utf-8-sig')
    cols = {_norm(c): c for c in df.columns}
    c_conc = next(cols[k] for k in cols if 'concurso' in k)
    c_data = next(cols[k] for k in cols if k.startswith('data'))
    bolas = sorted([k for k in cols if re.fullmatch(r'(bola|dezena)\d', k)], key=lambda k: int(k[-1]))
    if len(bolas) != 6:
        raise ValueError(f"Não achei as 6 colunas de bolas. Colunas: {list(df.columns)}")
    out = df[[c_conc, c_data] + [cols[b] for b in bolas]].copy()
    out.columns = ['concurso', 'data'] + [f'b{i}' for i in range(1, 7)]
    out = out.dropna(subset=['concurso', 'b1'])
    out['concurso'] = out['concurso'].astype(int)
    for i in range(1, 7):
        out[f'b{i}'] = out[f'b{i}'].astype(int)
    out['data'] = pd.to_datetime(out['data'], dayfirst=True, errors='coerce')
    # arquivos antigos da Caixa repetem o concurso quando há mais de uma cidade ganhadora
    dup = out.duplicated('concurso', keep=False)
    if dup.any():
        consist = out[dup].groupby('concurso')[[f'b{i}' for i in range(1, 7)]].nunique().max().max() == 1
        assert consist, "Concurso repetido com dezenas diferentes"
        out = out.drop_duplicates('concurso')
    return out.sort_values('concurso').reset_index(drop=True)


df = carregar(ARQUIVO)
if CORTE is not None:
    df = df[df['concurso'] <= CORTE].reset_index(drop=True)
D = df[[f'b{i}' for i in range(1, 7)]].to_numpy()

# --- checagens de integridade ---
assert D.min() >= 1 and D.max() <= DEZENAS, "Dezena fora de 1-60"
assert all(len(set(r)) == 6 for r in D), "Concurso com dezena repetida"
assert (df['concurso'].diff().dropna() == 1).all() and df['concurso'].iloc[0] == 1, "Há concursos faltando"

T = len(df)
HITS = np.zeros((T, DEZENAS), bool)
HITS[np.arange(T)[:, None], D - 1] = True

print("=" * 70)
print("  DADOS · MEGA-SENA, HISTÓRICO DE SORTEIOS")
print("=" * 70)
print(f"  Concursos:          {T:,}".replace(',', '.'))
print(f"  Do concurso 1 ({df['data'].iloc[0]:%d/%m/%Y}) ao {df['concurso'].iloc[-1]} ({df['data'].iloc[-1]:%d/%m/%Y})")
print(f"  Bolas sorteadas:    {T*6:,}".replace(',', '.'))
print(f"  Duas gaiolas:       concursos 1 a {CONCURSO_GLOBO_UNICO-1:,}".replace(',', '.'))
print(f"  Globo único:        concursos {CONCURSO_GLOBO_UNICO:,} a {df['concurso'].iloc[-1]:,}".replace(',', '.'))
print(f"\n  Chance de uma dezena sair num concurso:  6/60 = {P_DEZENA:.0%}")
print(f"  Chance da sena, aposta simples:          1 em {mil(COMBINACOES)}")
print(f"  De cada R$ {APOSTA_SIMPLES:.2f} apostados, voltam em prêmios: R$ {APOSTA_SIMPLES*PREMIO_BRUTO:.2f}".replace('.', ','))
print("=" * 70)

In [ ]:
# --- O MODELO ---
# Um único motor calcula as mesmas estatísticas para a série real e para
# os históricos simulados, em que cada concurso tira 6 de 60 sem reposição.


def motor(T, R, hits_reais=None, rng=None):
    nA, nQ = len(ROT_ATRASO), len(ROT_QUENTE)
    last = np.full((R, DEZENAS), -1)
    janela = np.zeros((R, JANELA, DEZENAS), bool)
    wsum = np.zeros((R, DEZENAS), int)
    freq = np.zeros((R, DEZENAS), int)
    maxgap = np.zeros(R, int)
    nA_ = np.zeros((R, nA)); xA_ = np.zeros((R, nA))
    nQ_ = np.zeros((R, nQ)); xQ_ = np.zeros((R, nQ))
    lin = np.arange(R)[:, None]
    for t in range(T):
        if hits_reais is not None:
            h = hits_reais[t][None, :]
        else:
            idx = rng.random((R, DEZENAS)).argpartition(SORTEADAS, axis=1)[:, :SORTEADAS]
            h = np.zeros((R, DEZENAS), bool); h[lin, idx] = True
        if t >= BURN:
            ok = last >= 0
            atraso = t - 1 - last
            bA = np.digitize(atraso, FAIXAS_ATRASO) - 1
            bQ = np.digitize(wsum, FAIXAS_QUENTE) - 1
            for b in range(nA):
                m = ok & (bA == b); nA_[:, b] += m.sum(1); xA_[:, b] += (m & h).sum(1)
            for b in range(nQ):
                m = (bQ == b); nQ_[:, b] += m.sum(1); xQ_[:, b] += (m & h).sum(1)
        gap = np.where(h & (last >= 0), t - last - 1, 0).max(1)
        maxgap = np.maximum(maxgap, gap)
        last[h] = t
        freq += h
        pos = t % JANELA
        wsum += h.astype(int) - janela[:, pos, :]
        janela[:, pos, :] = h
    atual = T - 1 - last
    return dict(freq=freq, maxgap=np.maximum(maxgap, atual.max(1)), atual_max=atual.max(1),
                atual=atual, nA=nA_, xA=xA_, nQ=nQ_, xQ=xQ_)


def x2_freq(freq, T):
    E = T * P_DEZENA
    return ((freq - E) ** 2 / E).sum(axis=-1)


def x2_homog(n, x):
    p = x.sum(-1, keepdims=True) / n.sum(-1, keepdims=True)
    return ((x - n * p) ** 2 / (n * p * (1 - p))).sum(-1)


def p_mc(sim, real):
    return (1 + (sim >= real).sum()) / (1 + len(sim))


real = motor(T, 1, hits_reais=HITS)
sim = motor(T, R_SIM, rng=rng)

# ---------------------------------------------------------------------------
print("=" * 70)
print("  PARTE 1 · A TEORIA: A BOLA NÃO TEM MEMÓRIA")
print("=" * 70)
print(f"  P(sair no próximo | atraso k) = 6/60 = {P_DEZENA:.0%}, para qualquer k")
print(f"  P(atraso ≥ k) = 0,9^k")
for k in (10, 20, 30, 42, 60):
    print(f"    k = {k:>2}:  {0.9**k:6.2%}".replace('.', ','))

# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("  PARTE 2 · A MEGA-SENA LEMBRA QUEM ESTÁ ATRASADO?")
print("=" * 70)
nA, xA = real['nA'][0], real['xA'][0]
jeff = [stats.beta(x + 0.5, n - x + 0.5) for n, x in zip(nA, xA)]
ic_atraso = np.array([b.interval(0.95) for b in jeff])
print(f"  {'Atraso':>8} {'Ocasiões':>10} {'Saiu':>7} {'Taxa':>7}   IC 95% (Beta de Jeffreys)")
for r, n, x, ic in zip(ROT_ATRASO, nA, xA, ic_atraso):
    print(f"  {r:>8} {mil(n):>10} {mil(x):>7} {x/n:>7.1%}   [{ic[0]:.1%}, {ic[1]:.1%}]")
X2A_real = x2_homog(real['nA'], real['xA'])[0]
X2A_sim = x2_homog(sim['nA'], sim['xA'])
pA = p_mc(X2A_sim, X2A_real)
print(f"\n  Teste de homogeneidade entre as faixas: X² = {X2A_real:.2f}, p (Monte Carlo) = {pA:.3f}")

print("\n  E quem está quente? Taxa de saída pela quantidade de vezes nos últimos 20:")
nQ, xQ = real['nQ'][0], real['xQ'][0]
ic_quente = np.array([stats.beta(x + 0.5, n - x + 0.5).interval(0.95) for n, x in zip(nQ, xQ)])
for r, n, x, ic in zip(ROT_QUENTE, nQ, xQ, ic_quente):
    print(f"  {r:>8} {mil(n):>10} {mil(x):>7} {x/n:>7.1%}   [{ic[0]:.1%}, {ic[1]:.1%}]")
X2Q_real = x2_homog(real['nQ'], real['xQ'])[0]
pQ = p_mc(x2_homog(sim['nQ'], sim['xQ']), X2Q_real)
print(f"  Teste de homogeneidade: X² = {X2Q_real:.2f}, p (Monte Carlo) = {pQ:.3f}")

# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("  PARTE 3 · O GLOBO É JUSTO? (uniformidade das 60 dezenas)")
print("=" * 70)
freq = real['freq'][0]
E = T * P_DEZENA
X2F_real = x2_freq(freq, T)
pF = p_mc(x2_freq(sim['freq'], T), X2F_real)
print(f"  Esperado por dezena: {E:.1f}")
print(f"  Série inteira: X² = {X2F_real:.1f}, p (Monte Carlo) = {pF:.3f}")
eras = {}
for nome, sl in [('Duas gaiolas', slice(0, CONCURSO_GLOBO_UNICO - 1)),
                 ('Globo único', slice(CONCURSO_GLOBO_UNICO - 1, T))]:
    Te = HITS[sl].shape[0]
    fe = HITS[sl].sum(0)
    se = motor(Te, R_SIM, rng=rng)
    eras[nome] = (Te, x2_freq(fe, Te), p_mc(x2_freq(se['freq'], Te), x2_freq(fe, Te)))
    print(f"  {nome:<13} ({mil(Te)} concursos): X² = {eras[nome][1]:.1f}, p (Monte Carlo) = {eras[nome][2]:.3f}")
    z = (fe - Te * P_DEZENA) / np.sqrt(Te * P_DEZENA * (1 - P_DEZENA))
    fora = [(d + 1, int(fe[d]), z[d]) for d in np.argsort(-np.abs(z)) if abs(z[d]) >= 3]
    print(f"      esperado {Te*P_DEZENA:.1f} por dezena · dezenas com |z| ≥ 3: " +
          (", ".join(f"{d:02d} ({n}, z = {zz:+.1f})" for d, n, zz in fora) if fora else "nenhuma"))
    eras[nome] += (fora,)

# checagem exploratória: o desvio se repete nas duas metades da era do globo único?
g = HITS[CONCURSO_GLOBO_UNICO - 1:]
h = len(g) // 2
print("\n  Exploratório · globo único dividido em duas metades:")
for nome, parte in [('1ª metade', g[:h]), ('2ª metade', g[h:])]:
    fe = parte.sum(0); Te = len(parte)
    print(f"    {nome} ({Te} concursos): X² = {x2_freq(fe, Te):.1f} · " +
          ", ".join(f"{d:02d}: {fe[d-1]}" for d in (np.argsort(-HITS[CONCURSO_GLOBO_UNICO-1:].sum(0))[:2] + 1)) +
          f" (esperado {Te*P_DEZENA:.0f})")

# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("  PARTE 4 · O ACASO PARECE ARRUMADO")
print("=" * 70)
ordem = np.argsort(-freq)
top, fundo = ordem[:3] + 1, ordem[-3:][::-1] + 1
sim_max, sim_min = sim['freq'].max(1), sim['freq'].min(1)
faixa = lambda a: (np.percentile(a, 5), np.percentile(a, 95))
print(f"  Mais sorteadas:  " + ", ".join(f"{d:02d} ({freq[d-1]})" for d in top))
print(f"  Menos sorteadas: " + ", ".join(f"{d:02d} ({freq[d-1]})" for d in fundo))
print(f"  Campeã real: {freq.max()} vezes · acaso puro produz campeã entre {faixa(sim_max)[0]:.0f} e {faixa(sim_max)[1]:.0f} (90%)")
print(f"  Lanterna real: {freq.min()} vezes · acaso puro produz lanterna entre {faixa(sim_min)[0]:.0f} e {faixa(sim_min)[1]:.0f} (90%)")
print(f"  Distância campeã-lanterna: real {freq.max()-freq.min()} · acaso {faixa(sim_max-sim_min)[0]:.0f} a {faixa(sim_max-sim_min)[1]:.0f}")
atual = real['atual'][0]
d_atr = np.argmax(atual) + 1
print(f"\n  Mais atrasada no fim da série: {d_atr:02d}, há {atual.max()} concursos sem sair")
print(f"  Acaso puro: a mais atrasada do momento está entre {faixa(sim['atual_max'])[0]:.0f} e {faixa(sim['atual_max'])[1]:.0f} (90%), mediana {np.median(sim['atual_max']):.0f}")
print(f"  Maior jejum da história real: {real['maxgap'][0]} concursos")
print(f"  Acaso puro: maior jejum entre {faixa(sim['maxgap'])[0]:.0f} e {faixa(sim['maxgap'])[1]:.0f} (90%), mediana {np.median(sim['maxgap']):.0f}")

# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("  PARTE 5 · TRÊS JOGADORES, A HISTÓRIA INTEIRA")
print("=" * 70)
rng_bt = np.random.default_rng(SEED + 1)
last = np.full(DEZENAS, -1); f = np.zeros(DEZENAS)
acertos = {'Atrasados': [], 'Mais sorteados': [], 'Surpresinha': []}
for t in range(T):
    if t >= BURN:
        ruido = rng_bt.random(DEZENAS) * 1e-6
        atr = np.where(last >= 0, t - 1 - last, t)
        jogos = {'Atrasados': np.argsort(-(atr + ruido))[:6],
                 'Mais sorteados': np.argsort(-(f + ruido))[:6],
                 'Surpresinha': rng_bt.choice(DEZENAS, 6, replace=False)}
        for k, j in jogos.items():
            acertos[k].append(HITS[t, j].sum())
    last[HITS[t]] = t; f += HITS[t]
n_bt = T - BURN
esperado = SORTEADAS * SORTEADAS / DEZENAS
print(f"  {mil(n_bt)} concursos jogados por cada um · esperado por puro acaso: {esperado:.2f} acertos por jogo")
bt = {}
for k, a in acertos.items():
    a = np.array(a); m = a.mean(); ep = a.std(ddof=1) / np.sqrt(len(a))
    q4, q5, q6 = (a == 4).sum(), (a == 5).sum(), (a == 6).sum()
    bt[k] = (m, m - 1.96 * ep, m + 1.96 * ep, q4, q5, q6)
    print(f"  {k:<15} média {m:.3f} [{m-1.96*ep:.3f}, {m+1.96*ep:.3f}] · quadras {q4} · quinas {q5} · senas {q6}")
pq = stats.hypergeom(DEZENAS, SORTEADAS, SORTEADAS)
print(f"  Esperado em {mil(n_bt)} jogos: {n_bt*pq.pmf(4):.1f} quadras · {n_bt*pq.pmf(5):.2f} quinas · {n_bt*pq.pmf(6):.5f} senas")

# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("  RESUMO PARA O INSIGHT")
print("=" * 70)
i30 = ROT_ATRASO.index('30-49')
print(f"  Com 30 a 49 concursos de atraso, a dezena saiu em {xA[i30]/nA[i30]:.1%} das vezes [{ic_atraso[i30][0]:.1%}, {ic_atraso[i30][1]:.1%}]")
print(f"  Com 50+ de atraso: {xA[-1]/nA[-1]:.1%} [{ic_atraso[-1][0]:.1%}, {ic_atraso[-1][1]:.1%}] em {int(nA[-1])} ocasiões")
print(f"  Memória do atraso: p = {pA:.2f} · memória do quente: p = {pQ:.2f}")
print(f"  Uniformidade: série p = {pF:.3f} · duas gaiolas p = {eras['Duas gaiolas'][2]:.3f} · globo único p = {eras['Globo único'][2]:.3f}")
print(f"  Campeã {freq.max()} (acaso: {faixa(sim_max)[0]:.0f}-{faixa(sim_max)[1]:.0f}) · lanterna {freq.min()} (acaso: {faixa(sim_min)[0]:.0f}-{faixa(sim_min)[1]:.0f})")
print(f"  Mais atrasada no fim da série: {d_atr:02d} com {atual.max()} · maior jejum: {real['maxgap'][0]}")
print(f"  Atrasados {bt['Atrasados'][0]:.3f} · Mais sorteados {bt['Mais sorteados'][0]:.3f} · Surpresinha {bt['Surpresinha'][0]:.3f} · teoria {esperado:.2f}")

In [ ]:
# --- VISUALIZAÇÃO ---
rodape = lambda txt: plt.figtext(0.5, 0.005, txt + ' | #365Probabilidades', ha='center', fontsize=9, color='gray')
FONTE = f"Fonte: Caixa Econômica Federal, concursos 1 a {df['concurso'].iloc[-1]} · Monte Carlo com {R_SIM} históricos"

# ── GRÁFICO 1 · A bola não tem memória ──
fig1, ax1 = plt.subplots(figsize=(11, 6.5))
xs = np.arange(len(ROT_ATRASO))
taxa = xA / nA
ax1.axhline(P_DEZENA * 100, color=DOURADO, lw=2, ls='--', label='Teoria: 6/60 = 10%, para qualquer atraso')
ax1.errorbar(xs, taxa * 100, yerr=[(taxa - ic_atraso[:, 0]) * 100, (ic_atraso[:, 1] - taxa) * 100],
             fmt='o', color=VERDE, ms=10, capsize=6, lw=2, label='Mega-Sena real (IC 95%, Beta de Jeffreys)')
for x_, t_, n_ in zip(xs, taxa, nA):
    ax1.text(x_, ic_atraso[x_, 1] * 100 + 0.35, f"{t_:.1%}".replace('.', ','), ha='center', fontsize=11, fontweight='bold')
    ax1.text(x_, 3.3, f"n = {int(n_):,}".replace(',', '.'), ha='center', fontsize=9, color=CINZA)
ax1.set_xticks(xs, ROT_ATRASO)
ax1.set_xlabel('Há quantos concursos a dezena não saía')
ax1.set_ylabel('Saiu no concurso seguinte (%)')
ax1.set_ylim(3, 17)
ax1.set_title(f'A bola não tem memória\nP(sair | atraso k) = 10% · teste entre faixas: p = {pA:.2f}'.replace('.', ','), fontsize=13, pad=12)
ax1.legend(loc='upper left', frameon=False)
rodape(FONTE)
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-104-grafico-01-sem-memoria.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ── GRÁFICO 2 · O acaso parece arrumado ──
fig2, ax2 = plt.subplots(figsize=(11, 6.5))
rank = np.arange(1, DEZENAS + 1)
sim_sorted = -np.sort(-sim['freq'], axis=1)
lo, hi = np.percentile(sim_sorted, 5, axis=0), np.percentile(sim_sorted, 95, axis=0)
ax2.fill_between(rank, lo, hi, color=CINZA, alpha=0.25, label='O que o acaso puro produz (90% dos históricos)')
ax2.plot(rank, np.median(sim_sorted, axis=0), color=CINZA, lw=1, ls=':')
ax2.plot(rank, freq[ordem], 'o-', color=VERMELHO, ms=4, lw=1.5, label='Ranking real das 60 dezenas')
ax2.axhline(E, color=DOURADO, lw=1.5, ls='--', label=f'Esperado: {E:.0f} vezes cada'.replace('.', ','))
for i in (0, 1, 2, DEZENAS - 3, DEZENAS - 2, DEZENAS - 1):
    d = ordem[i] + 1
    ax2.annotate(f"{d:02d}", (i + 1, freq[d - 1]), textcoords='offset points',
                 xytext=(0, 9 if i < 30 else -15), ha='center', fontsize=10, fontweight='bold')
ax2.set_xlabel('Posição no ranking (1 = mais sorteada)')
ax2.set_ylabel('Vezes que saiu')
ax2.set_title(f'O acaso parece arrumado\nRanking real das 60 dezenas contra {R_SIM} sorteios justos · uniformidade: p = {pF:.3f}'.replace('.', ','), fontsize=13, pad=12)
ax2.legend(loc='upper right', frameon=False)
rodape(FONTE)
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-104-grafico-02-ranking.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ── GRÁFICO 3 · A mais atrasada é o que o acaso produz ──
fig3, (a, b) = plt.subplots(1, 2, figsize=(12, 6))
for ax, dados, valor, tit in [(a, sim['atual_max'], atual.max(), 'A mais atrasada do momento'),
                              (b, sim['maxgap'], real['maxgap'][0], 'O maior jejum da história')]:
    ax.hist(dados, bins=np.arange(dados.min(), dados.max() + 3, 2) - 0.5, color=VERDE, alpha=0.55, edgecolor='white')
    ax.axvline(valor, color=VERMELHO, lw=2.5)
    p5, p95 = np.percentile(dados, [5, 95])
    ax.axvspan(p5, p95, color=DOURADO, alpha=0.12)
    ax.text(valor, ax.get_ylim()[1] * 0.92, f'  real: {valor}', color=VERMELHO, fontsize=11, fontweight='bold')
    ax.set_title(tit, fontsize=12)
    ax.set_xlabel('Concursos sem sair')
a.set_ylabel(f'Históricos simulados (de {R_SIM})')
fig3.suptitle('A dezena "atrasada" é o que um sorteio justo produz\n' + r'$P(\mathrm{atraso} \geq k) = 0{,}9^{k}$' + ' · faixa dourada: 90% dos históricos de acaso puro', fontsize=13)
rodape(FONTE)
plt.tight_layout(rect=(0, 0.03, 1, 0.93))
plt.savefig('dia-104-grafico-03-atraso.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

### 💡 O Insight

**A Mega-Sena não lembra.**

Quando uma dezena estava há 30 a 49 concursos sem sair, ela saiu no concurso seguinte em **9,5% das vezes**, em 6.956 ocasiões. Com 50 ou mais de atraso, em 9,7%. A teoria diz 10%, e é isso que 3.005 concursos mostram. O teste entre as faixas de atraso dá p = 0,53: nenhum sinal de que o atraso mude qualquer coisa.

Com os números "quentes" é igual. Ter saído muito nos últimos 20 concursos não muda a chance do próximo (p = 0,22).

A dezena mais atrasada no fim da série é a **25, há 52 concursos sem aparecer**. Parece um recado. Para uma dezena específica, um jejum desse tamanho tem 0,4% de chance. Só que são 60 dezenas disputando esse posto, e num sorteio perfeitamente justo a mais atrasada do momento costuma estar entre 29 e 66 concursos sem sair. O atraso que chama atenção é o comportamento normal de um acaso sem memória.

Os três jogadores terminam empatados. Em 2.985 concursos, quem apostou sempre nas seis mais atrasadas acertou em média **0,58** dezena por jogo. Quem apostou nas seis mais sorteadas, **0,60**. A Surpresinha, **0,60**. A teoria manda 0,60 para os três, e nenhum deles fez uma quina em quase 3 mil tentativas.

Mas tem uma parte que não fechou, e ela merece ser dita.

Uma coisa é o globo não ter memória. Outra é ele tratar as 60 dezenas exatamente igual, e esse segundo teste não passou. Nos concursos a partir do 1.140, quando as duas gaiolas foram trocadas por um globo único, as frequências ficam mais desiguais do que 2.000 sorteios justos produziriam (p = 0,001). A **dezena 10 saiu 240 vezes, contra 187 esperadas**, e o desvio se repete nas duas metades desse período. Na era das duas gaiolas, nada disso aparece.

Não sei a causa, e um teste que rejeita não aponta culpado: pode ser o jogo de bolas, o equipamento, ou algo na própria transcrição dos resultados. O que dá para dizer é que a diferença é pequena demais para virar estratégia e grande demais para eu fingir que não vi.

Nada disso aproxima ninguém do prêmio. A chance da sena numa aposta simples continua sendo 1 em 50.063.860.

O que muda é outra coisa. Se o globo não guarda nada, escolher pela lista dos atrasados não é estratégia, é superstição com aparência de planilha. E aí você fica livre para jogar no número que quiser, inclusive naquele que só você entende.

*Qual número você ainda joga por achar que ele está devendo?*

---

### ⚠️ Limitações do Modelo

- **O arquivo não é o download oficial da Caixa**, e sim uma cópia pública dos mesmos resultados, no mesmo formato. As checagens de integridade garantem 6 dezenas distintas entre 1 e 60 em cada concurso e nenhum concurso faltando na sequência, mas não substituem a conferência contra a fonte oficial. Qualquer erro de transcrição na cópia apareceria como desvio de uniformidade, e é por isso que o achado da dezena 10 fica como pergunta em aberto, não como conclusão.
- **A série vai até o concurso 3.005, de 07/05/2026**, e não até a véspera da publicação.
- **A data da troca de equipamento (concurso 1.140) vem da Wikipédia** e não foi confirmada em documento da Caixa. Se estiver errada, só o teste por era muda. Os testes de memória usam a série inteira.
- **"Memória" aqui significa atraso e frequência recente.** Pares que saem juntos, somas, sequências e padrões de dia da semana não foram testados.
- **Os intervalos de Jeffreys por faixa tratam cada ocasião como independente.** Como as dezenas de um mesmo concurso competem entre si, eles ficam um pouco estreitos demais. Os p-valores dos testes vêm do Monte Carlo do mecanismo real e não têm esse problema.
- **Se o globo não for perfeitamente uniforme, as dezenas que saem menos acumulam atrasos maiores.** Isso empurra a taxa das faixas de atraso longo para baixo, no sentido contrário ao da crença.
- **A divisão da era do globo único em duas metades é exploratória**, não foi definida antes de rodar.
- **Os três jogadores usam estratégias simples.** Não cobrem todos os sistemas vendidos como "método".
- **Nada aqui é recomendação de aposta.** Pelas regras vigentes, de cada R$ 6,00 apostados, R$ 2,63 voltam em prêmios.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
